In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import joblib
from sklearn.metrics import (
    accuracy_score, classification_report,
    roc_auc_score, roc_curve, confusion_matrix, matthews_corrcoef, auc
)
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import train_test_split
from imblearn.pipeline import Pipeline
import numpy as np

from Feature_Extraction import compute_aac, compute_dpc, compute_tpc, compute_pcp, one_hot_encode

import os
os.makedirs("Metrics/MLP", exist_ok=True)
os.makedirs("Saved Models/MLP", exist_ok=True)

import pandas as pd
from sklearn.model_selection import GroupShuffleSplit          # replaces train_test_split

df = pd.read_csv("balanced_named.csv")

X = df.drop(columns=["label", "sequence", "cluster"])          # keep the two new columns out of the features
y = df["label"]
groups = df["cluster"]                                         # +

feature_names = X.columns      # used later by the SHAP section

# random split: cluster membership is deliberately ignored
train_idx, test_idx = train_test_split(
    np.arange(len(df)), test_size=0.2, stratify=y, random_state=42)

# each cluster goes entirely to train or entirely to test
#splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
#train_idx, test_idx = next(splitter.split(X, y, groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print("features:", X.shape[1], "(should be 9036)")
print("train:", len(y_train), " test:", len(y_test))
print("test label balance:\n", y_test.value_counts())

features: 9036 (should be 9036)
train: 5292  test: 1324
test label balance:
 label
1    662
0    662
Name: count, dtype: int64


In [18]:
pipe_mlp = Pipeline([
    ("select", SelectKBest(score_func=f_classif, k=20)),
    ("scale",  StandardScaler()),
    ("clf",    MLPClassifier(
        hidden_layer_sizes=(100, 100, 50),
        alpha=0.0001,
        batch_size=64,
        learning_rate_init=0.00166,
        random_state=42,
        early_stopping=True,
        validation_fraction=0.1,
        tol=1e-4
    ))
])

# =============
# Fit and Predict
# =============
pipe_mlp.fit(X_train, y_train)
#joblib.dump(pipe_mlp, "Saved Models\\MLP\\mlp_pipeline.pkl")
y_pred = pipe_mlp.predict(X_test)
y_prob = pipe_mlp.predict_proba(X_test)[:, 1]

# ======================
# Classification Metrics
# ======================
print("=== MLP Stand-Alone Model ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:\n", classification_report(y_test, y_pred, digits=4))

# ===================
# Confusion Matrix Plot
# ===================
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("MLP - Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.savefig("Metrics\\MLP\\mlp_confusion_matrix.png")
plt.close()

# ==========
# ROC Curve
# ==========
fpr, tpr, _ = roc_curve(y_test, y_prob)
plt.figure(figsize=(6, 4))
plt.plot(fpr, tpr, label=f"AUC = {roc_auc_score(y_test, y_prob):.4f}")
plt.plot([0, 1], [0, 1], "k--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("MLP - ROC Curve")
plt.legend()
plt.tight_layout()
plt.savefig("Metrics\\MLP\\mlp_roc_curve.png")
plt.close()

# =============
# SHAP Analysis
# =============
# Apply transformations manually for SHAP input
selector = pipe_mlp.named_steps["select"]
scaler = pipe_mlp.named_steps["scale"]
clf = pipe_mlp.named_steps["clf"]

X_sample = X_test[:100]
X_selected = selector.transform(X_sample)
X_scaled = scaler.transform(X_selected)

# Feature names
selected_feature_names = np.array(feature_names)[selector.get_support()]

# Use unified SHAP Explainer
explainer = shap.Explainer(clf.predict_proba, X_scaled)
shap_values = explainer(X_scaled)

# SHAP Summary Plot
plt.figure()
mean_shap = np.mean(np.abs(shap_values.values), axis=2)  # shape: (100, 20)
# Plot mean SHAP values across classes
shap.summary_plot(
    mean_shap,
    features=X_scaled,
    feature_names=selected_feature_names,
    plot_type="bar",
    max_display=20,
    show=False
)

plt.tight_layout()
plt.savefig("Metrics/MLP/mlp_shap_summary_combined_classes.png")
plt.close()

c:\Users\sadaf\anaconda3\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:111: UserWarning: Features [ 639  678  718  786  798  870  958  978 1018 1026 1030 1037 1038 1039
 1070 1090 1098 1192 1193 1218 1318 1378 1426 1430 1458 1510 1513 1558
 1581 1586 1701 1718 1758 1838 1938 1980 1981 1986 1998 2050 2058 2098
 2158 2178 2226 2227 2230 2239 2258 2290 2310 2370 2378 2383 2390 2396
 2397 2398 2399 2406 2838 2850 2918 2958 2970 2991 3020 3030 3035 3036
 3038 3050 3070 3098 3118 3150 3159 3178 3180 3186 3189 3194 3195 3196
 3197 3198 3199 3350 3430 3438 3441 3484 3593 3824 3826 4158 4178 4224
 4298 4381 4398 4446 4450 4459 4464 4472 4473 4478 4498 4512 4518 4538
 4541 4546 4550 4551 4552 4553 4554 4555 4558 4559 4590 4598 4621 4626
 4627 4637 4638 4644 4670 4678 4685 4698 4706 4718 4730 4758 4766 4778
 4782 4784 4789 4790 4792 4793 4796 4798 4799 4804 4806 4810 4811 4817
 4818 5038 5070 5110 5183 5186 5198 5250 5258 5298 5358 5438 5610 5618
 5659 5706 5758 5821 5831 5

=== MLP Stand-Alone Model ===
Accuracy: 0.9350

Classification Report:
               precision    recall  f1-score   support

           0     0.9311    0.9396    0.9353       662
           1     0.9390    0.9305    0.9347       662

    accuracy                         0.9350      1324
   macro avg     0.9351    0.9350    0.9350      1324
weighted avg     0.9351    0.9350    0.9350      1324



C:\Users\sadaf\AppData\Local\Temp\ipykernel_24680\1105664095.py:82: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(


In [19]:
# ========================================
# External Validation on Benchmark Dataset
# ========================================

import pandas as pd
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             matthews_corrcoef, roc_auc_score, average_precision_score)

external = pd.read_csv("external_benchmark.csv")

ext_X = pd.DataFrame(
    [compute_aac(s) + compute_dpc(s) + compute_tpc(s) + compute_pcp(s)
     + list(one_hot_encode(s)) for s in external["sequence"]],
    columns=X.columns, dtype="float32")
ext_y = external["label"].to_numpy()

ext_prob = pipe_mlp.predict_proba(ext_X)[:, 1]
ext_pred = (ext_prob >= 0.5).astype(int)

tn, fp, fn, tp = confusion_matrix(ext_y, ext_pred, labels=[0, 1]).ravel()

print("=== MLP — External Benchmark ===")
print(f"Accuracy:    {accuracy_score(ext_y, ext_pred):.4f}")
print(f"MCC:         {matthews_corrcoef(ext_y, ext_pred):.4f}")
print(f"ROC-AUC:     {roc_auc_score(ext_y, ext_prob):.4f}")
print(f"PR-AUC:      {average_precision_score(ext_y, ext_prob):.4f}")
print(f"Sensitivity: {tp/(tp+fn):.4f}")
print(f"Specificity: {tn/(tn+fp):.4f}")
print("\nClassification Report:\n", classification_report(ext_y, ext_pred, digits=4))

=== MLP — External Benchmark ===
Accuracy:    0.7540
MCC:         0.5178
ROC-AUC:     0.8472
PR-AUC:      0.8654
Sensitivity: 0.8506
Specificity: 0.6575

Classification Report:
               precision    recall  f1-score   support

           0     0.8148    0.6575    0.7278       803
           1     0.7129    0.8506    0.7757       803

    accuracy                         0.7540      1606
   macro avg     0.7639    0.7540    0.7517      1606
weighted avg     0.7639    0.7540    0.7517      1606



In [14]:
# ================
# Repeated Seeds
# ================
from sklearn.base import clone

accs, mccs = [], []
for seed in range(10):
    tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
                  .split(X, y, groups))
    p = clone(pipe_mlp)                      # change to your pipeline name
    p.fit(X.iloc[tr], y.iloc[tr])
    pred = p.predict(X.iloc[te])
    accs.append(accuracy_score(y.iloc[te], pred))
    mccs.append(matthews_corrcoef(y.iloc[te], pred))
    print(f"seed {seed}: acc={accs[-1]:.4f}  MCC={mccs[-1]:.4f}")

print(f"\nacc {np.mean(accs):.4f} ± {np.std(accs, ddof=1):.4f}")
print(f"MCC {np.mean(mccs):.4f} ± {np.std(mccs, ddof=1):.4f}")

c:\Users\sadaf\anaconda3\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:111: UserWarning: Features [ 621  638  782  786  798  853  870  906  958  978 1018 1024 1026 1030
 1037 1038 1039 1098 1189 1192 1193 1218 1348 1426 1458 1510 1511 1513
 1558 1581 1586 1701 1718 1758 1826 1980 1981 1986 1992 1998 2018 2050
 2051 2070 2141 2158 2178 2226 2227 2230 2258 2290 2310 2378 2383 2387
 2390 2398 2399 2413 2838 2850 2918 2942 2951 2963 2970 2991 3020 3030
 3035 3036 3038 3050 3070 3098 3150 3153 3178 3180 3185 3186 3194 3195
 3196 3197 3198 3199 3216 3258 3298 3350 3438 3441 3510 3592 3824 3826
 4158 4178 4298 4381 4398 4446 4450 4458 4459 4464 4472 4473 4498 4512
 4518 4538 4541 4550 4551 4552 4553 4554 4558 4559 4570 4590 4598 4621
 4627 4629 4636 4638 4639 4650 4666 4678 4698 4706 4718 4719 4730 4739
 4758 4778 4782 4784 4789 4790 4792 4793 4796 4798 4799 4801 4804 4809
 4810 4811 4815 4818 4850 5038 5070 5110 5186 5198 5250 5258 5298 5346
 5350 5356 5358 5430 5438 5

seed 0: acc=0.9057  MCC=0.8107


c:\Users\sadaf\anaconda3\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:111: UserWarning: Features [ 718  786  798  870  878  958  978 1018 1022 1026 1030 1037 1038 1039
 1098 1106 1170 1186 1192 1193 1213 1218 1318 1351 1426 1458 1510 1511
 1513 1558 1581 1586 1590 1701 1718 1758 1904 1980 1981 1986 1998 2050
 2158 2178 2226 2227 2230 2258 2290 2310 2378 2383 2387 2389 2390 2397
 2398 2399 2838 2850 2870 2878 2918 2970 2991 3020 3030 3035 3036 3038
 3050 3059 3070 3098 3136 3150 3156 3178 3180 3186 3194 3195 3196 3197
 3198 3199 3210 3298 3350 3430 3438 3441 3824 3826 3986 3996 4006 4158
 4178 4298 4381 4398 4446 4447 4450 4459 4464 4472 4473 4478 4498 4512
 4518 4538 4541 4550 4551 4552 4553 4554 4558 4559 4590 4598 4621 4627
 4638 4644 4650 4658 4678 4685 4698 4706 4718 4719 4730 4758 4773 4778
 4782 4784 4790 4791 4792 4793 4796 4798 4799 4804 4807 4810 4811 4818
 4819 4950 5038 5070 5110 5186 5198 5218 5250 5298 5358 5421 5426 5438
 5570 5610 5618 5659 5758 5

seed 1: acc=0.9405  MCC=0.8811


c:\Users\sadaf\anaconda3\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:111: UserWarning: Features [ 639  718  786  798  870  958  978 1018 1022 1024 1026 1030 1037 1038
 1039 1090 1098 1106 1192 1193 1194 1213 1218 1426 1430 1458 1510 1513
 1558 1581 1586 1701 1718 1758 1838 1858 1980 1981 1986 1990 1998 2050
 2158 2178 2226 2227 2230 2239 2250 2258 2290 2310 2378 2383 2390 2396
 2398 2399 2838 2850 2858 2918 2970 2991 3020 3026 3030 3032 3035 3036
 3038 3050 3070 3084 3092 3094 3098 3150 3178 3180 3186 3187 3194 3195
 3196 3197 3198 3199 3350 3438 3441 3510 3706 3824 3826 4158 4178 4298
 4381 4398 4446 4447 4450 4459 4464 4472 4473 4478 4498 4504 4510 4512
 4518 4538 4541 4546 4550 4551 4552 4553 4554 4558 4559 4590 4598 4621
 4626 4627 4638 4639 4640 4666 4678 4679 4685 4698 4706 4718 4719 4726
 4730 4746 4750 4758 4761 4778 4780 4782 4783 4784 4790 4792 4793 4796
 4798 4799 4804 4806 4810 4811 4818 5026 5032 5038 5070 5110 5186 5198
 5246 5250 5298 5346 5350 5

seed 2: acc=0.9213  MCC=0.8428


... (10 outputs left out)

c:\Users\sadaf\anaconda3\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:111: UserWarning: Features [ 782  786  798  858  870  878  958  978 1018 1026 1030 1037 1038 1039
 1098 1106 1184 1192 1193 1218 1318 1426 1446 1458 1510 1513 1558 1581
 1586 1701 1718 1758 1838 1839 1858 1980 1981 1986 1990 1992 1998 2050
 2058 2070 2090 2141 2158 2178 2226 2227 2229 2230 2258 2290 2310 2378
 2383 2390 2397 2398 2399 2838 2850 2918 2958 2970 2991 3020 3030 3035
 3036 3038 3050 3070 3098 3118 3133 3146 3150 3159 3178 3180 3186 3187
 3194 3195 3196 3197 3198 3199 3218 3253 3258 3298 3350 3438 3441 3590
 3599 3601 3824 3826 3834 4158 4178 4298 4381 4387 4398 4446 4450 4459
 4464 4472 4473 4498 4506 4512 4518 4538 4541 4542 4544 4545 4550 4551
 4552 4553 4554 4556 4558 4559 4590 4598 4606 4621 4624 4627 4637 4638
 4678 4679 4684 4698 4706 4718 4730 4758 4761 4778 4782 4784 4789 4790
 4792 4793 4796 4798 4799 4804 4810 4811 4816 4818 5038 5070 5110 5186
 5198 5250 5298 5343 5346 5

seed 8: acc=0.9329  MCC=0.8673


c:\Users\sadaf\anaconda3\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:111: UserWarning: Features [ 632  638  718  786  798  870  878  958  978 1018 1024 1026 1030 1037
 1038 1039 1046 1084 1098 1192 1193 1212 1218 1426 1458 1510 1513 1558
 1581 1586 1701 1718 1758 1826 1858 1938 1980 1981 1982 1986 1998 2050
 2110 2141 2144 2153 2158 2178 2224 2226 2227 2229 2230 2234 2239 2258
 2290 2310 2378 2383 2390 2398 2399 2406 2443 2558 2786 2832 2838 2850
 2893 2918 2970 2991 3020 3023 3030 3035 3036 3038 3050 3070 3090 3098
 3099 3107 3150 3158 3178 3180 3185 3186 3189 3194 3195 3196 3197 3198
 3199 3210 3244 3350 3438 3441 3459 3822 3824 3826 3834 3996 4018 4158
 4178 4224 4238 4298 4381 4398 4399 4446 4450 4459 4464 4472 4473 4478
 4498 4508 4512 4513 4518 4538 4540 4541 4548 4550 4551 4552 4553 4554
 4555 4558 4559 4570 4590 4598 4599 4621 4627 4638 4678 4698 4706 4718
 4730 4758 4778 4782 4784 4790 4792 4793 4796 4798 4799 4804 4806 4807
 4810 4811 4818 4901 4950 5

seed 9: acc=0.9450  MCC=0.8899

acc 0.9230 ± 0.0145
MCC 0.8464 ± 0.0290


In [11]:
import numpy as np
from sklearn.base import clone
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, matthews_corrcoef

accs, mccs = [], []
for seed in range(10):
    tr, te = train_test_split(np.arange(len(df)), test_size=0.2,
                              stratify=y, random_state=seed)
    p = clone(pipe_mlp)
    p.fit(X.iloc[tr], y.iloc[tr])
    pred = p.predict(X.iloc[te])
    accs.append(accuracy_score(y.iloc[te], pred))
    mccs.append(matthews_corrcoef(y.iloc[te], pred))
    print(f"seed {seed}: acc={accs[-1]:.4f}  MCC={mccs[-1]:.4f}")

print(f"\nacc {np.mean(accs):.4f} ± {np.std(accs, ddof=1):.4f}")
print(f"MCC {np.mean(mccs):.4f} ± {np.std(mccs, ddof=1):.4f}")

c:\Users\sadaf\anaconda3\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:111: UserWarning: Features [ 544  786  798  863  870  958  978 1018 1026 1030 1037 1038 1039 1098
 1106 1150 1186 1192 1193 1218 1426 1439 1458 1510 1513 1558 1581 1586
 1701 1718 1758 1980 1981 1986 1998 2018 2050 2158 2178 2224 2226 2227
 2230 2258 2290 2310 2370 2378 2383 2390 2398 2399 2478 2838 2850 2901
 2918 2938 2970 2991 3020 3030 3035 3036 3038 3050 3070 3098 3150 3159
 3178 3180 3185 3186 3194 3195 3196 3197 3198 3199 3210 3253 3350 3424
 3438 3441 3584 3824 3826 3838 3986 4158 4178 4224 4298 4381 4387 4398
 4446 4450 4459 4464 4472 4473 4498 4512 4518 4538 4541 4548 4550 4551
 4552 4553 4554 4558 4559 4590 4598 4621 4624 4627 4636 4638 4639 4678
 4698 4706 4718 4730 4758 4778 4782 4784 4786 4790 4792 4793 4796 4798
 4799 4804 4808 4810 4811 4818 4856 4950 5026 5038 5070 5110 5186 5198
 5218 5250 5298 5358 5430 5438 5610 5618 5659 5744 5758 5822 5826 5839
 5970 5983 6226 6229 6236 6

seed 0: acc=0.9275  MCC=0.8550


c:\Users\sadaf\anaconda3\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:111: UserWarning: Features [ 786  798  870  898  926  958  977  978 1018 1026 1030 1037 1038 1039
 1090 1098 1192 1193 1218 1426 1458 1461 1510 1513 1558 1581 1586 1701
 1718 1758 1838 1980 1981 1982 1983 1986 1998 2050 2058 2158 2178 2224
 2226 2227 2230 2246 2258 2290 2310 2378 2383 2384 2387 2389 2390 2396
 2398 2399 2698 2786 2838 2850 2918 2938 2942 2958 2970 2991 3020 3023
 3024 3030 3035 3036 3038 3050 3070 3098 3118 3150 3178 3180 3185 3186
 3194 3195 3196 3197 3198 3199 3216 3258 3298 3350 3424 3438 3441 3490
 3586 3590 3824 3826 4158 4178 4218 4298 4381 4398 4446 4450 4459 4464
 4472 4473 4492 4498 4510 4512 4518 4538 4541 4546 4550 4551 4552 4553
 4554 4558 4559 4590 4598 4621 4624 4626 4627 4638 4658 4670 4678 4685
 4698 4706 4718 4719 4730 4739 4747 4758 4778 4782 4783 4784 4789 4790
 4791 4792 4793 4796 4798 4799 4804 4807 4810 4811 4818 4850 5032 5038
 5070 5110 5184 5186 5198 5

seed 1: acc=0.9358  MCC=0.8717


c:\Users\sadaf\anaconda3\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:111: UserWarning: Features [ 638  678  786  798  870  940  958  978 1018 1024 1026 1030 1037 1038
 1039 1098 1192 1193 1218 1426 1439 1458 1510 1511 1513 1558 1581 1586
 1701 1718 1758 1838 1839 1980 1981 1986 1996 1998 2018 2050 2098 2158
 2178 2224 2226 2227 2230 2236 2258 2290 2310 2370 2378 2383 2390 2398
 2399 2826 2838 2850 2901 2918 2958 2970 2991 3020 3030 3035 3036 3038
 3050 3070 3098 3150 3178 3180 3186 3189 3190 3194 3195 3196 3197 3198
 3199 3310 3350 3438 3441 3484 3824 3826 3838 4158 4178 4224 4226 4298
 4381 4392 4398 4446 4450 4459 4464 4472 4473 4498 4504 4512 4518 4538
 4541 4546 4550 4551 4552 4553 4554 4555 4556 4558 4559 4590 4598 4618
 4621 4624 4626 4627 4638 4670 4678 4698 4706 4718 4719 4726 4730 4758
 4778 4782 4783 4784 4786 4790 4792 4793 4796 4798 4799 4804 4808 4810
 4811 4818 5026 5038 5070 5110 5186 5198 5250 5258 5298 5358 5438 5490
 5500 5610 5618 5659 5719 5

seed 2: acc=0.9373  MCC=0.8747


... (10 outputs left out)

c:\Users\sadaf\anaconda3\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:111: UserWarning: Features [ 544  786  798  870  958  978 1018 1026 1030 1033 1037 1038 1039 1070
 1098 1192 1193 1210 1218 1348 1426 1458 1510 1513 1558 1581 1582 1586
 1701 1718 1758 1839 1980 1981 1986 1998 2050 2158 2178 2226 2227 2230
 2258 2290 2310 2378 2383 2389 2390 2398 2399 2698 2838 2850 2918 2951
 2970 2991 3020 3030 3032 3035 3036 3038 3050 3064 3066 3070 3098 3150
 3178 3180 3186 3187 3194 3195 3196 3197 3198 3199 3216 3258 3350 3424
 3430 3438 3441 3581 3592 3599 3824 3826 3986 4018 4158 4178 4224 4238
 4298 4359 4381 4398 4446 4450 4459 4464 4472 4473 4498 4508 4512 4518
 4538 4539 4541 4550 4551 4552 4553 4554 4558 4559 4570 4590 4598 4621
 4627 4638 4650 4658 4678 4698 4706 4718 4730 4758 4778 4782 4784 4787
 4790 4792 4793 4796 4798 4799 4804 4810 4811 4818 5032 5038 5070 5110
 5186 5198 5250 5258 5298 5358 5359 5428 5438 5610 5618 5659 5718 5758
 5766 5835 5970 5982 5983 5

seed 8: acc=0.9396  MCC=0.8792


c:\Users\sadaf\anaconda3\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:111: UserWarning: Features [ 678  786  798  870  958  978 1018 1024 1026 1030 1033 1037 1038 1039
 1098 1106 1192 1193 1210 1213 1218 1426 1430 1458 1510 1513 1558 1581
 1582 1586 1594 1701 1718 1758 1798 1938 1980 1981 1986 1998 2050 2118
 2119 2153 2158 2178 2226 2227 2230 2236 2258 2290 2310 2378 2383 2389
 2390 2398 2399 2418 2838 2850 2918 2938 2970 2991 3020 3030 3035 3036
 3038 3050 3070 3098 3107 3150 3153 3178 3180 3186 3187 3190 3194 3195
 3196 3197 3198 3199 3216 3258 3350 3378 3438 3441 3484 3590 3599 3650
 3751 3824 3826 4018 4158 4178 4298 4381 4387 4398 4446 4447 4450 4459
 4464 4472 4473 4498 4512 4513 4518 4538 4541 4550 4551 4552 4553 4554
 4558 4559 4590 4598 4604 4621 4627 4638 4658 4678 4679 4698 4706 4718
 4730 4758 4778 4782 4784 4789 4790 4792 4793 4796 4798 4799 4804 4810
 4811 4818 5038 5070 5110 5186 5198 5250 5298 5358 5438 5500 5590 5610
 5618 5659 5706 5758 5766 5

seed 9: acc=0.9162  MCC=0.8323

acc 0.9297 ± 0.0118
MCC 0.8595 ± 0.0236
